# AIOps Silver Feature Build

Purpose: create ML-ready feature tables from the silver Delta surfaces used by the GE/Spark rule-based pipeline. 

In [0]:
from pyspark.sql import functions as F

STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

BASE = f'abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net'
RUN_ID = spark.sql("select date_format(current_timestamp(), 'yyyyMMddHHmmss') run_id").first()['run_id']

TRAINING_SILVER_HEADER_PATH = f'{BASE}/training/silver/training_invoice_cleansed/header/'
TRAINING_SILVER_LINES_PATH = f'{BASE}/training/silver/training_invoice_cleansed/lines/'

FEATURE_ROOT = f'{BASE}/monitoring/aiops/features'
SILVER_HEADER_FEATURE_PATH = f'{FEATURE_ROOT}/training/header/'
SILVER_LINES_FEATURE_PATH = f'{FEATURE_ROOT}/training/lines/'

print(f'Silver feature build run_id={RUN_ID}')

Silver feature build run_id=20260423105850


In [0]:
def read_delta(path, label):
    df = spark.read.format('delta').load(path)
    if df.rdd.isEmpty():
        raise RuntimeError(f'{label} is empty: {path}')
    return df

def num(df, name):
    return F.col(name).cast('double') if name in df.columns else F.lit(None).cast('double')

def present(df, name):
    return F.when(F.col(name).isNotNull(), 1.0).otherwise(0.0) if name in df.columns else F.lit(0.0)

def write_delta(df, path):
    (df.withColumn('feature_run_id', F.lit(RUN_ID))
       .withColumn('feature_build_ts', F.current_timestamp())
       .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(path))
    print(path)

In [0]:
header = read_delta(TRAINING_SILVER_HEADER_PATH, 'silver_header')
lines = read_delta(TRAINING_SILVER_LINES_PATH, 'silver_lines')

line_rollup = (lines.groupBy('InvoiceId')
               .agg(
        F.count('LineNumber').cast('double').alias('line_count'),
        F.sum(num(lines, 'ItemSubTotal')).cast(
            'double').alias('sum_line_subtotal'),
        F.avg(num(lines, 'Quantity')).cast('double').alias('avg_quantity'),
        F.avg(num(lines, 'UnitPrice')).cast('double').alias('avg_unit_price'),
        F.max(num(lines, 'ItemSubTotal')).cast('double').alias('max_line_subtotal'),
        F.stddev(num(lines, 'ItemSubTotal')).cast('double').alias('stddev_line_subtotal')
    ))

header_features = (header.join(line_rollup, 'InvoiceId', 'left')
                   .select('InvoiceId', F.col('_source_file').cast('string').alias('SourceFile'), F.col('source_type').cast('string').alias('SourceType'),
                           present(header, 'InvoiceId').alias(
                               'invoice_id_present'),
                           present(header, 'OrderDate').alias(
                               'order_date_present'),
                           present(header, 'CustomerName').alias(
                               'customer_name_present'),
                           present(header, 'ShipMode').alias(
                               'ship_mode_present'),
                           F.when(F.col('source_type') == 'csv', 1.0).when(
                           F.col('source_type') == 'json', 2.0).otherwise(0.0).alias('source_type_code'),
        F.abs(F.xxhash64(F.coalesce(F.col('ShipMode'), F.lit('missing'))) %
                           1000).cast('double').alias('ship_mode_code'),
        num(header, 'BalanceDue').alias('BalanceDue'), num(
                           header, 'SubTotal').alias('SubTotal'),
        num(header, 'DiscountPercent').alias('DiscountPercent'), num(
                           header, 'DiscountAmount').alias('DiscountAmount'),
        num(header, 'ShippingAmount').alias('ShippingAmount'), num(
                           header, 'InvoiceTotal').alias('InvoiceTotal'),
        F.coalesce(F.col('line_count'), F.lit(0.0)).alias('line_count'), F.coalesce(
                           F.col('sum_line_subtotal'), F.lit(0.0)).alias('sum_line_subtotal'),
        F.coalesce(F.col('avg_quantity'), F.lit(0.0)).alias('avg_quantity'), F.coalesce(F.col('avg_unit_price'), F.lit(0.0)).alias('avg_unit_price'),
        F.coalesce(F.col('max_line_subtotal'), F.lit(0.0)).alias('max_line_subtotal'),
        F.coalesce(F.col('stddev_line_subtotal'), F.lit(0.0)).alias('stddev_line_subtotal'))
        .withColumn('calc_invoice_total', F.col('SubTotal') - F.coalesce(F.col('DiscountAmount'), F.lit(0.0)) + F.coalesce(F.col('ShippingAmount'), F.lit(0.0)))
        .withColumn('header_total_diff', F.col('InvoiceTotal') - F.col('calc_invoice_total'))
        .withColumn('header_subtotal_rollup_diff', F.col('SubTotal') - F.col('sum_line_subtotal'))
        .withColumn('discount_ratio', F.when(F.col('SubTotal') != 0, F.col('DiscountAmount') / F.col('SubTotal')).otherwise(0.0))
        .withColumn('missing_header_field_count', (1.0 - F.col('invoice_id_present')) + (1.0 - F.col('order_date_present')) + (1.0 - F.col('customer_name_present')) + (1.0 - F.col('ship_mode_present')))
        .withColumn('missing_financial_field_count',
                    F.when(F.col('BalanceDue').isNull(), 1.0).otherwise(0.0)
                    + F.when(F.col('SubTotal').isNull(), 1.0).otherwise(0.0)
                    + F.when(F.col('DiscountPercent').isNull(), 1.0).otherwise(0.0)
                    + F.when(F.col('DiscountAmount').isNull(), 1.0).otherwise(0.0)
                    + F.when(F.col('ShippingAmount').isNull(), 1.0).otherwise(0.0)
                    + F.when(F.col('InvoiceTotal').isNull(), 1.0).otherwise(0.0))
        .withColumn('abs_header_total_diff', F.abs(F.coalesce(F.col('header_total_diff'), F.lit(0.0))))
        .withColumn('abs_header_subtotal_rollup_diff', F.abs(F.coalesce(F.col('header_subtotal_rollup_diff'), F.lit(0.0))))
        .withColumn('shipping_ratio', F.when(F.col('SubTotal') != 0, F.col('ShippingAmount') / F.col('SubTotal')).otherwise(0.0))
        .withColumn('balance_due_ratio', F.when(F.col('InvoiceTotal') != 0, F.col('BalanceDue') / F.col('InvoiceTotal')).otherwise(0.0))
        .withColumn('line_count_log', F.log1p(F.abs(F.coalesce(F.col('line_count'), F.lit(0.0)))))
        .withColumn('avg_line_amount', F.when(F.col('line_count') != 0, F.col('sum_line_subtotal') / F.col('line_count')).otherwise(0.0)))

write_delta(header_features, SILVER_HEADER_FEATURE_PATH)


abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/monitoring/aiops/features/training/header/


In [0]:
header_invoice_total = header.select(
    'InvoiceId',
    num(header, 'InvoiceTotal').alias('_invoice_total')
)

line_source = lines.join(header_invoice_total, 'InvoiceId', 'left')

line_features = (line_source.select('InvoiceId', F.col('LineNumber').cast('string').alias('LineNumber'),
            F.col('_source_file').cast('string').alias('SourceFile'), F.col('source_type').cast('string').alias('SourceType'),
            present(line_source, 'InvoiceId').alias('invoice_id_present'), present(line_source, 'LineNumber').alias('line_number_present'),
            present(line_source, 'ProductName').alias('product_name_present'), present(line_source, 'ProductId').alias('product_id_present'),
            present(line_source, 'Quantity').alias('quantity_present'), present(line_source, 'UnitPrice').alias('unit_price_present'),
            present(line_source, 'ItemSubTotal').alias('item_subtotal_present'),
            F.when(F.col('source_type') == 'csv', 1.0).when(F.col('source_type') == 'json', 2.0).otherwise(0.0).alias('source_type_code'),
            F.abs(F.xxhash64(F.coalesce(F.col('ProductName'), F.lit('missing'))) % 1000).cast('double').alias('product_name_code'),
            num(line_source, 'Quantity').alias('Quantity'), num(line_source, 'UnitPrice').alias('UnitPrice'), num(line_source, 'ItemSubTotal').alias('ItemSubTotal'),
            F.coalesce(F.col('_invoice_total'), F.lit(0.0)).alias('_invoice_total'))
        .withColumn('calc_item_subtotal', F.col('Quantity') * F.col('UnitPrice'))
        .withColumn('line_subtotal_diff', F.col('ItemSubTotal') - F.col('calc_item_subtotal'))
        .withColumn('abs_line_subtotal_diff', F.abs(F.coalesce(F.col('line_subtotal_diff'), F.lit(0.0))))
        .withColumn('line_amount_ratio_to_invoice', F.when(F.col('_invoice_total') != 0, F.col('ItemSubTotal') / F.col('_invoice_total')).otherwise(0.0))
        .withColumn('quantity_log', F.log1p(F.abs(F.coalesce(F.col('Quantity'), F.lit(0.0)))))
        .withColumn('unit_price_log', F.log1p(F.abs(F.coalesce(F.col('UnitPrice'), F.lit(0.0)))))
        .withColumn('item_subtotal_log', F.log1p(F.abs(F.coalesce(F.col('ItemSubTotal'), F.lit(0.0)))))
        .drop('_invoice_total'))

write_delta(line_features, SILVER_LINES_FEATURE_PATH)
dbutils.jobs.taskValues.set(key='aiops_silver_header_feature_path', value=SILVER_HEADER_FEATURE_PATH)
dbutils.jobs.taskValues.set(key='aiops_silver_lines_feature_path', value=SILVER_LINES_FEATURE_PATH)


abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/monitoring/aiops/features/training/lines/
